# Notebook to run nerfstudio experiments
- Add new experiments by appending to `EXPERIMENTS`
- Add any new CLI flag by putting it in `extra_args` (gets mapped to `--flag value`)
- Add nested method args via `method_args`
- Set `DRY_RUN = True` to preview commands without executing

# Imports

In [ ]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime

import os
import subprocess
import sys
import time
import itertools
import shutil
from itertools import product

# Workspace Setup

In [ ]:
# Check conda environment
env_name = os.environ.get("CONDA_DEFAULT_ENV", "unknown")
print(f"Conda environment: {env_name}")
print(f"Python: {sys.executable}")
print(f"ns-train found: {shutil.which('ns-train')}")

# For cleaner logs because nerfstudio uses rich for terminal output
os.environ["NO_COLOR"] = "1"
os.environ["TERM"] = "dumb"

# Paths
WORKSPACE_DIR = "/home/saber/workspaces/irwin_ws/fyp-playground"
DATASETS = {
    "torpedo_unprocessed": os.path.join(WORKSPACE_DIR, "datasets", "torpedo", "torpedo_unprocessed"),
    "saltpond_unprocessed": os.path.join(WORKSPACE_DIR, "datasets", "saltpond", "saltpond_unprocessed"),
}
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")
LOG_DIR = os.path.join(WORKSPACE_DIR, "logs")

os.chdir(WORKSPACE_DIR)
print(f"Working directory: {os.getcwd()}")

# Experiments Configuration

In [ ]:
# Experiment templates - model and params only (independent of dataset)
EXPERIMENT_TEMPLATES = [
    {   # control experiment to check default settings
        "suffix": "aa_delete",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": 0.005,
            "pipeline.model.use-scale-regularization": True,
        },
     }
]

# * Grid searches
for seathru_iter, update_backscatter_at_interval, update_backscatter_at_count in product(
    [5_000, 10_000, 15_000],    # seathru-from-iter
    [50, 100, 200],             # update-backscatter-at-interval
    [25, 50, 100],              # update-backscatter-at-count
):
    EXPERIMENT_TEMPLATES.append({
        "suffix": "optimization-schedule",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": 0.005,
            "pipeline.model.use-scale-regularization": True,
            "pipeline.model.seathru-from-iter": seathru_iter,
            "pipeline.model.update-backscatter-at-interval": update_backscatter_at_interval,
            "pipeline.model.update-backscatter-at-count": update_backscatter_at_count,
        },
    })


for cull_alpha_thresh, use_scale_regularization in product(
    [0.005, 0.01, 0.05, 0.1, 0.5],
    [True, False]
):
    EXPERIMENT_TEMPLATES.append({
        "suffix": "sweep_cull-alpha-thresh_scale-regularization",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": cull_alpha_thresh,
            "pipeline.model.use-scale-regularization": use_scale_regularization,
        },
     })


for bg_lambda in [0.001, 0.01, 0.05, 0.1]:
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_bg-lambda",
        "extra_args": {
            "pipeline.model.bg-lambda": bg_lambda,
        },
     })
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_bg-lambda",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": 0.005,
            "pipeline.model.use-scale-regularization": False,
            "pipeline.model.bg-lambda": bg_lambda,
        },
     })
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_bg-lambda",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": 0.005,
            "pipeline.model.use-scale-regularization": True,
            "pipeline.model.bg-lambda": bg_lambda,
        },
     })
    
for dcp_loss_lambda in [0.1, 0.5, 1.0, 5.0]:
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_dcp-loss-lambda",
        "extra_args": {
            "pipeline.model.dcp-loss-lambda": dcp_loss_lambda,
        },
     })
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_dcp-loss-lambda",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": 0.005,
            "pipeline.model.use-scale-regularization": False,
            "pipeline.model.dcp-loss-lambda": dcp_loss_lambda,
        },
     })
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_dcp-loss-lambda",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": 0.005,
            "pipeline.model.use-scale-regularization": True,
            "pipeline.model.dcp-loss-lambda": dcp_loss_lambda,
        },
     })
    
for gw_loss_lambda in [0.01, 0.05, 0.1, 0.5]:
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_gw-loss-lambda",
        "extra_args": {
            "pipeline.model.gw-loss-lambda": gw_loss_lambda,
        },
     })
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_gw-loss-lambda",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": 0.005,
            "pipeline.model.use-scale-regularization": False,
            "pipeline.model.gw-loss-lambda": gw_loss_lambda,
        },
     })
    EXPERIMENT_TEMPLATES.append({
        "suffix": f"sweep_gw-loss-lambda",
        "extra_args": {
            "pipeline.model.cull-alpha-thresh": 0.005,
            "pipeline.model.use-scale-regularization": True,
            "pipeline.model.gw-loss-lambda": gw_loss_lambda,
        },
     })

# Base configs
base_steps_per_eval_image = 1000
base_steps_per_eval_batch = 0
base_steps_per_save = 2000
base_steps_per_eval_all_images = 1000
base_max_iters = 30000

EXPERIMENTS = []
for dataset_name, dataset_path in DATASETS.items():
    print(f"{dataset_path} exists: {Path(dataset_path).exists()}")
    for template in EXPERIMENT_TEMPLATES:
        name = f"{dataset_name}/{template['suffix']}"
        EXPERIMENTS.append({
            "name": name,
            "model": "sea-splatfacto",
            "data": dataset_path,
            "output_dir": os.path.join(OUTPUT_DIR, dataset_name),
            "viewer": False,
            "extra_args": {
                "steps-per-eval-image": template.get("steps_per_eval_image", base_steps_per_eval_image),
                "steps-per-eval-batch": template.get("steps_per_eval_batch", base_steps_per_eval_batch),
                "steps-per-eval-all-images": template.get("steps_per_eval_all_images", base_steps_per_eval_all_images),
                "steps-per-save": template.get("steps_per_save", base_steps_per_save),
                "max-num-iterations": template.get("max_iters", base_max_iters),
                "experiment-name": f"{dataset_name}-{template['suffix']}",
                **template["extra_args"],
            },
        })

DRY_RUN = False  # start with True to preview commands first

# To run subsets

In [ ]:
# Only one dataset
EXPERIMENTS = [e for e in EXPERIMENTS if "saltpond_unprocessed" in e["name"]]

# Only baselines across all datasets
# EXPERIMENTS = [e for e in EXPERIMENTS if e["name"].endswith("baseline")]

# Specific combo
# EXPERIMENTS = [e for e in EXPERIMENTS if e["name"] == "red-sea-wreck/no-priors"]

# Multiple specific templates
# KEEP = {"baseline", "no-seathru", "backscatter-only"}
# EXPERIMENTS = [e for e in EXPERIMENTS if e["name"].split("/")[1] in KEEP]

print(f"Running {len(EXPERIMENTS)} experiments")

# Command builder

In [ ]:
def build_command(experiment: dict) -> list[str]:
    """Build an ns-train CLI command from an experiment config dict.

    Any key in extra_args automatically becomes a --flag value pair.
    """
    cmd = ["ns-train", experiment["model"]]

    cmd += ["--data", str(experiment["data"])]

    output_dir = experiment.get("output_dir", "./outputs")
    cmd += ["--output-dir", str(output_dir)]

    if experiment.get("viewer") is False:
        cmd += ["--viewer.quit-on-train-completion", "True"]

    for key, value in experiment.get("extra_args", {}).items():
        cmd += [f"--{key}", str(value)]

    for key, value in experiment.get("method_args", {}).items():
        cmd += [f"--{key}", str(value)]

    return cmd


def command_to_str(cmd: list[str]) -> str:
    """Pretty-print a command list as a shell string."""
    return " ".join(cmd)

# Preview commands

In [ ]:
for i, exp in enumerate(EXPERIMENTS):
    cmd = build_command(exp)
    print(f"[{i}] {exp['name']}")
    print(f"    {command_to_str(cmd)}")
    print()

# Run experiments

In [ ]:
results = []

for i, exp in enumerate(EXPERIMENTS):
    cmd = build_command(exp)
    name = exp["name"]
    print(f"[{i+1}/{len(EXPERIMENTS)}] Starting: {name}")
    
    if DRY_RUN:
        results.append({"name": name, "status": "skipped (dry run)", "duration": 0})
        continue
    
    Path(exp.get("output_dir", "./outputs")).mkdir(parents=True, exist_ok=True)
    dataset_name = name.split("/")[0]
    log_subdir = Path(LOG_DIR) / dataset_name
    log_subdir.mkdir(parents=True, exist_ok=True)
    log_file = log_subdir / f"{exp['extra_args']['experiment-name']}.log"
    
    start = time.time()
    try:
        with open(log_file, "w") as f:
            proc = subprocess.run(
                cmd,
                check=True,
                text=True,
                stdout=f,
                stderr=subprocess.STDOUT,
            )
        status = "success"
    except subprocess.CalledProcessError as e:
        status = f"failed (exit code {e.returncode})"
        # Save failed log with a unique name
        failed_log_file = log_subdir / f"{i}_{exp['extra_args']['experiment-name']}_FAILED_{int(time.time())}.log"
        shutil.copy2(log_file, failed_log_file)
        log_file = failed_log_file  # Update log_file reference for results
    except FileNotFoundError:
        status = "failed (ns-train not found)"
        failed_log_file = log_subdir / f"{i}_{exp['extra_args']['experiment-name']}_FAILED_{int(time.time())}.log"
        if log_file.exists():
            shutil.copy2(log_file, failed_log_file)
            log_file = failed_log_file
    
    duration = time.time() - start
    results.append({"name": name, "status": status, "duration": duration, "log": str(log_file)})
    print(f"  → {status} ({duration/60:.1f} min) — log: {log_file}")

# Summary

In [ ]:
print(f"\n{'Experiment':<40} {'Status':<25} {'Duration':>10}")
print("-" * 75)
for r in results:
    dur = f"{r['duration']/60:.1f} min" if r['duration'] > 0 else "—"
    print(f"{r['name']:<40} {r['status']:<25} {dur:>10}")